In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, GridBox
from IPython.display import display

# ============================================================
# CSS FOR HORIZONTAL RADIO BUTTONS
# ============================================================

display(HTML("""
<style>
.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    align-items: center !important;
    gap: 18px !important;
}
.horizontal-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}
.horizontal-radio > label {
    display: none !important;
}
</style>
"""))

# ============================================================
# INTERACTIVE FUNCTION
# ============================================================

def plot_correlation_transformation(input_type='White noise', system_type='Low-pass', rho=0.80, alpha=0.75):

    # --------------------------------------------------------
    # LAG AXIS
    # --------------------------------------------------------

    K = 80
    k = np.arange(-K, K + 1)

    # ========================================================
    # INPUT AUTOCORRELATION Rxx[k]
    # ========================================================

    if input_type == 'White noise':

        Rxx = np.zeros_like(k, dtype=float)
        Rxx[K] = 1.0

        input_name = 'White Noise'

    else:

        Rxx = rho ** np.abs(k)

        input_name = f'Correlated AR(1), ρ = {rho:.2f}'

    # ========================================================
    # IMPULSE RESPONSE h[n]
    # ========================================================

    M = 60
    n = np.arange(0, M + 1)

    if system_type == 'Low-pass':

        h = (1.0 - alpha) * alpha ** n

        system_name = f'Low-Pass, α = {alpha:.2f}'

    else:

        h = np.zeros(M + 1)

        h[0] = 1.0
        h[1] = -1.0

        system_name = 'First-Difference High-Pass'

    # ========================================================
    # REVERSED IMPULSE RESPONSE h[-n]
    # ========================================================

    h_rev = h[::-1]

    # ========================================================
    # Rxy[k] = h[-k] * Rxx[k]
    # ========================================================

    Rxy_full = signal.convolve(Rxx, h_rev, mode='full')

    lags_xy = np.arange(
        k[0] - M,
        k[-1] + 1
    )

    # ========================================================
    # g[k] = h[k] * h[-k]
    # ========================================================

    g = signal.convolve(
        h,
        h_rev,
        mode='full'
    )

    # ========================================================
    # Ryy[k] = g[k] * Rxx[k]
    # ========================================================

    Ryy_full = signal.convolve(
        Rxx,
        g,
        mode='full'
    )

    lags_yy = np.arange(
        k[0] - M,
        k[-1] + M + 1
    )

    # ========================================================
    # EXTRACT COMMON LAG INTERVAL [-K, K]
    # ========================================================

    mask_xy = (
        (lags_xy >= -K)
        & (lags_xy <= K)
    )

    k_xy = lags_xy[mask_xy]
    Rxy = Rxy_full[mask_xy]

    mask_yy = (
        (lags_yy >= -K)
        & (lags_yy <= K)
    )

    k_yy = lags_yy[mask_yy]
    Ryy = Ryy_full[mask_yy]

    # ========================================================
    # NORMALIZATION FOR DISPLAY
    # ========================================================

    Rxx_plot = Rxx.copy()
    Rxy_plot = Rxy.copy()
    Ryy_plot = Ryy.copy()

    if np.max(np.abs(Rxy_plot)) > 0:

        Rxy_plot = Rxy_plot / np.max(np.abs(Rxy_plot))

    if np.max(np.abs(Ryy_plot)) > 0:

        Ryy_plot = Ryy_plot / np.max(np.abs(Ryy_plot))

    # ========================================================
    # FIGURE
    # ========================================================

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(11.3, 3.9))

    # ========================================================
    # GRAPH 1:
    # INPUT AUTOCORRELATION
    # ========================================================

    ax1.plot(k, Rxx_plot, linewidth=2)

    ax1.axhline(
        0,
        linewidth=0.8,
        color='black'
    )

    ax1.axvline(
        0,
        linewidth=0.8,
        linestyle=':',
        color='black'
    )

    ax1.set_xlim(
        -K,
        K
    )

    ax1.set_ylim(
        -1.15,
        1.15
    )

    ax1.set_xlabel(
        'Lag k',
        fontsize=11
    )

    ax1.set_ylabel(
        'Rₓₓ[k]',
        fontsize=11
    )

    ax1.set_title(
        f'Input Autocorrelation\n{input_name}',
        fontsize=12,
        pad=9
    )

    ax1.tick_params(
        axis='both',
        labelsize=9
    )

    ax1.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # GRAPH 2:
    # CROSS-CORRELATION
    # ========================================================

    ax2.plot(k_xy, Rxy_plot, linewidth=2)

    ax2.axhline(
        0,
        linewidth=0.8,
        color='black'
    )

    ax2.axvline(
        0,
        linewidth=0.8,
        linestyle=':',
        color='black'
    )

    ax2.set_xlim(
        -K,
        K
    )

    ax2.set_ylim(
        -1.15,
        1.15
    )

    ax2.set_xlabel(
        'Lag k',
        fontsize=11
    )

    ax2.set_ylabel(
        'Normalized Rₓᵧ[k]',
        fontsize=11
    )

    ax2.set_title(
        f'Input-Output Cross-Correlation\n{system_name}',
        fontsize=12,
        pad=9
    )

    ax2.tick_params(
        axis='both',
        labelsize=9
    )

    ax2.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # GRAPH 3:
    # OUTPUT AUTOCORRELATION
    # ========================================================

    ax3.plot(k_yy, Ryy_plot, linewidth=2)

    ax3.axhline(
        0,
        linewidth=0.8,
        color='black'
    )

    ax3.axvline(
        0,
        linewidth=0.8,
        linestyle=':',
        color='black'
    )

    ax3.set_xlim(
        -K,
        K
    )

    ax3.set_ylim(
        -1.15,
        1.15
    )

    ax3.set_xlabel(
        'Lag k',
        fontsize=11
    )

    ax3.set_ylabel(
        'Normalized Rᵧᵧ[k]',
        fontsize=11
    )

    ax3.set_title(
        'Output Autocorrelation',
        fontsize=12,
        pad=9
    )

    ax3.tick_params(
        axis='both',
        labelsize=9
    )

    ax3.grid(
        True,
        linestyle=':',
        alpha=0.5
    )

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    plt.subplots_adjust(
        left=0.06,
        right=0.98,
        top=0.82,
        bottom=0.18,
        wspace=0.32
    )

    plt.show()
    plt.close(fig)

# ============================================================
# RADIO BUTTONS
# ============================================================

input_selector = RadioButtons(
    options=['White noise', 'Correlated AR(1)'],
    value='White noise',
    description='',
    layout=Layout(width='255px')
)

input_selector.add_class(
    'horizontal-radio'
)

system_selector = RadioButtons(
    options=['Low-pass', 'High-pass'],
    value='Low-pass',
    description='',
    layout=Layout(width='220px')
)

system_selector.add_class(
    'horizontal-radio'
)

# ============================================================
# SLIDERS
# ============================================================

slider_style = {
    'description_width': '0px'
}

slider_layout = Layout(
    width='145px'
)

rho_slider = FloatSlider(
    min=0.0,
    max=0.98,
    step=0.02,
    value=0.80,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

alpha_slider = FloatSlider(
    min=0.10,
    max=0.95,
    step=0.05,
    value=0.75,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# CURRENT VALUE AND MAXIMUM VALUE LABELS
# ============================================================

rho_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91; width:42px;">0.80</div>'
)

alpha_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91; width:42px;">0.75</div>'
)

rho_max = HTML(
    '<div style="font-family:Arial; font-size:13px; color:#555555;">max 0.98</div>'
)

alpha_max = HTML(
    '<div style="font-family:Arial; font-size:13px; color:#555555;">max 0.95</div>'
)

# ============================================================
# UPDATE CURRENT VALUES
# ============================================================

def update_rho_value(change):

    rho_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91; width:42px;">{rho_slider.value:.2f}</div>'

def update_alpha_value(change):

    alpha_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91; width:42px;">{alpha_slider.value:.2f}</div>'

rho_slider.observe(
    update_rho_value,
    names='value'
)

alpha_slider.observe(
    update_alpha_value,
    names='value'
)

# ============================================================
# ENABLE / DISABLE SLIDERS
# ============================================================

def update_controls(change):

    rho_slider.disabled = (
        input_selector.value != 'Correlated AR(1)'
    )

    alpha_slider.disabled = (
        system_selector.value != 'Low-pass'
    )

input_selector.observe(
    update_controls,
    names='value'
)

system_selector.observe(
    update_controls,
    names='value'
)

# ============================================================
# INTERACTIVE OBJECT
# ============================================================

widget_plot = interactive(
    plot_correlation_transformation,
    input_type=input_selector,
    system_type=system_selector,
    rho=rho_slider,
    alpha=alpha_slider
)

# ============================================================
# DOCUMENTATION
# ============================================================

theory_html = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.45;
    width:1050px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#7a3e00;
    margin-bottom:9px;
">
Correlation Transformation Through an LTI System
</div>

<div style="margin-bottom:5px;">
<b>Input autocorrelation:</b> Rₓₓ[k] describes the statistical dependence between input samples separated by lag k.
</div>

<div style="margin-bottom:5px;">
<b>Cross-correlation:</b> Rₓᵧ[k] describes the statistical relationship between the input and the filtered output.
</div>

<div style="margin-bottom:5px;">
<b>Output autocorrelation:</b> Rᵧᵧ[k] describes the correlation structure produced at the output of the LTI system.
</div>

<div style="
    font-family:serif;
    font-size:20px;
    font-style:italic;
    color:#7a3e00;
    margin:8px 0px 8px 20px;
">
Rₓᵧ[k] = h*[−k] * Rₓₓ[k]
&nbsp;&nbsp;&nbsp;&nbsp;
Rᵧᵧ[k] = h[k] * h*[−k] * Rₓₓ[k]
</div>

<div>
<b>This notebook:</b> shows how an LTI system transforms the correlation structure of a WSS random input process.
</div>

</div>
""")

# ============================================================
# CONTROL LABELS
# ============================================================

input_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">Input process:</div>'
)

system_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">System:</div>'
)

rho_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">AR parameter ρ:</div>'
)

alpha_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold;">LP parameter α:</div>'
)

# ============================================================
# CONTROL ROWS
# ============================================================

input_box = HBox(
    [input_label, input_selector],
    layout=Layout(
        width='420px',
        align_items='center'
    )
)

system_box = HBox(
    [system_label, system_selector],
    layout=Layout(
        width='330px',
        align_items='center'
    )
)

rho_box = HBox(
    [rho_label, rho_slider, rho_value, rho_max],
    layout=Layout(
        width='390px',
        align_items='center'
    )
)

alpha_box = HBox(
    [alpha_label, alpha_slider, alpha_value, alpha_max],
    layout=Layout(
        width='390px',
        align_items='center'
    )
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HBox(
            [input_box, system_box],
            layout=Layout(
                width='900px',
                align_items='center'
            )
        ),
        HBox(
            [rho_box, alpha_box],
            layout=Layout(
                width='900px',
                align_items='center'
            )
        )
    ],
    layout=Layout(
        width='930px',
        padding='10px 14px',
        border='1px solid #d5c4b4',
        margin='14px 0px 8px 0px',
        overflow='hidden'
    )
)

# ============================================================
# INTERPRETATION
# ============================================================

interpretation_html = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1050px;
    padding:12px 16px;
    border:1px solid #dfc9ad;
    background:#fffaf4;
    box-sizing:border-box;
    margin-top:4px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#7a3e00;
    margin-bottom:7px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:5px;">
<b>Left:</b> Rₓₓ[k] describes the correlation already present in the stochastic input.
</div>

<div style="margin-bottom:5px;">
<b>Center:</b> Rₓᵧ[k] shows how the impulse response relates the input process to the output process.
</div>

<div style="margin-bottom:5px;">
<b>Right:</b> Rᵧᵧ[k] is the new correlation structure produced by the LTI system.
</div>

<div style="margin-top:8px;">
For <b>white noise</b>, Rₓₓ[k] is concentrated at k = 0, but after passage through an LTI system with memory, Rᵧᵧ[k] generally extends over nonzero lags.
</div>

<div style="
    margin-top:8px;
    font-weight:bold;
    color:#7a3e00;
">
Therefore, an LTI system with memory can transform an uncorrelated random input into a temporally correlated random output.
</div>

</div>
""")

# ============================================================
# COMPLETE LAYOUT
# ============================================================

main_layout = VBox(
    [
        theory_html,
        controls_card,
        widget_plot.children[-1],
        interpretation_html
    ],
    layout=Layout(
        width='1050px',
        overflow='hidden'
    )
)

display(main_layout)